# Tracking object - Kalman filter

In [13]:
import cv2
import matplotlib.pyplot as plt
import pandas as pd
import glob
import numpy as np
import time

### Functions

In [2]:

class KalmanFilter(object):
    """
        Kalman filter class. 
        Source tutorial: https://machinelearningspace.com/2d-object-tracking-using-kalman-filter/
    """
    
    def __init__(self, dt, u_x,u_y, std_acc, x_std_meas, y_std_meas):
        """
        dt: sampling time (time for 1 cycle)
        u_x: acceleration in x-direction
        u_y: acceleration in y-direction
        std_acc: process noise magnitude
        x_std_meas: standard deviation of the measurement in x-direction
        y_std_meas: standard deviation of the measurement in y-direction
        """

        # Define sampling time
        self.dt = dt

        # Define the  control input variables
        self.u = np.matrix([[u_x],[u_y]])

        # Intial State
        self.x = np.matrix([[0], [0], [0], [0]])

        # Define the State Transition Matrix A
        self.A = np.matrix([[1, 0, self.dt, 0],
                            [0, 1, 0, self.dt],
                            [0, 0, 1, 0],
                            [0, 0, 0, 1]])

        # Define the Control Input Matrix B
        self.B = np.matrix([[(self.dt**2)/2, 0],
                            [0,(self.dt**2)/2],
                            [self.dt,0],
                            [0,self.dt]])

        # Define Measurement Mapping Matrix
        self.H = np.matrix([[1, 0, 0, 0],
                            [0, 1, 0, 0]])

        #Initial Process Noise Covariance
        self.Q = np.matrix([[(self.dt**4)/4, 0, (self.dt**3)/2, 0],
                            [0, (self.dt**4)/4, 0, (self.dt**3)/2],
                            [(self.dt**3)/2, 0, self.dt**2, 0],
                            [0, (self.dt**3)/2, 0, self.dt**2]]) * std_acc**2

        #Initial Measurement Noise Covariance
        self.R = np.matrix([[x_std_meas**2,0],
                           [0, y_std_meas**2]])

        #Initial Covariance Matrix
        self.P = np.eye(self.A.shape[1])

    def predict(self):

        # Update time state
        self.x = np.dot(self.A, self.x) + np.dot(self.B, self.u)

        # Calculate error covariance
        self.P = np.dot(np.dot(self.A, self.P), self.A.T) + self.Q
        return self.x[0:2]

    def update(self, z):
        
        S = np.dot(self.H, np.dot(self.P, self.H.T)) + self.R

        # Calculate the Kalman Gain
        K = np.dot(np.dot(self.P, self.H.T), np.linalg.inv(S)) 
        self.x = np.round(self.x + np.dot(K, (z - np.dot(self.H, self.x)))) 
        I = np.eye(self.H.shape[1])

        # Update error covariance matrix
        self.P = (I - (K * self.H)) * self.P
        return self.x[0:2]
    

def create_video(images_lists, text_list, output_path, fps=24):
    # Get the height and width of the frames
    height, width = images_lists[0][0].shape[:2]

    # Define the codec and create a VideoWriter object
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    video_writer = cv2.VideoWriter(output_path, fourcc, fps, (2 * width, 2 * height), isColor=True)

    # Iterate through the frames in each list
    for i in range(len(images_lists[0])):
        # Create a 2 by 2 grid by concatenating images horizontally and vertically
        top_row = np.concatenate((images_lists[0][i], images_lists[1][i]), axis=1)
        bottom_row = np.concatenate((images_lists[2][i], images_lists[3][i]), axis=1)
        final_frame = np.concatenate((top_row, bottom_row), axis=0)
       
       
        # Add text to each space in the grid
        for j, text in enumerate(text_list):
            text_position = (width * (j % 2), height * (j // 2) + 20)
            cv2.putText(final_frame, str(text), text_position, cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,0,0), 1, cv2.LINE_AA)
 
        # Write the frame to the video file
        video_writer.write(final_frame)

    # Release the VideoWriter object
    video_writer.release()

        

    
def create_video(sequence, output_path, fps=24):
    # Get the height and width of the frames
    height, width = sequence[0].shape[:2]

    # Define the codec and create a VideoWriter object
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    video_writer = cv2.VideoWriter(output_path, fourcc, fps, (width, height), isColor=True)

    # Iterate through the frames in each list
    for frame in sequence:
        
        # Write the frame to the video file
        video_writer.write(frame)

    # Release the VideoWriter object
    video_writer.release()


    
def iou_coef(groundtruth_mask, pred_mask):
    """Calculate IoU coefficient for similariy
    given a ground truth and a prediction image"""
    
    intersect = np.sum(pred_mask*groundtruth_mask)
    union = np.sum(pred_mask) + np.sum(groundtruth_mask) - intersect
    iou = np.mean(intersect/union)
    if np.isnan(iou): # Ground truth and pred mask all zeros
        iou = 1.0
    return round(iou, 3)


def precision_score(groundtruth_mask, pred_mask):
    """Calculate precision score by pixel 
    given a ground truth and a prediction image"""
    
    intersect = np.sum(pred_mask*groundtruth_mask)
    total_pixel_pred = np.sum(pred_mask)
    precision = np.mean(intersect/total_pixel_pred)
    if np.isnan(precision): # Ground truth and pred mask all zeros
        precision = 1.0
    return round(precision, 3)

def accuracy_score(groundtruth_mask, pred_mask):
    """Calculate accuracy score by pixel 
    given a ground truth and a prediction image"""
    
    intersect = np.sum(pred_mask*groundtruth_mask)
    union = np.sum(pred_mask) + np.sum(groundtruth_mask) - intersect
    xor = np.sum(groundtruth_mask==pred_mask)
    acc = np.mean(xor/(union + xor - intersect))
    return round(acc, 3)


def extract_metrics(ground_truth, estimated):
    # Metrics
    precision_tot = []
    accuracy_tot = []
    iou_tot = []

    for gt, estim in zip(ground_truth, estimated):
        precision = precision_score(gt, estim)
        accuracy  = accuracy_score(gt, estim)
        iou = iou_coef(gt, estim)

        precision_tot.append(precision)
        accuracy_tot.append(accuracy)
        iou_tot.append(iou)

    print(f"Pixel Accuracy: {round(np.mean(accuracy_tot),3)}")
    print(f"Pixel Precision: {round(np.mean(precision_tot),3)}")
    print(f"IoU Metric: {round(np.mean(iou_tot),3)}")

### Loading data

In [3]:
# Ground truth
gt = pd.read_csv('gt.txt', sep=',').to_numpy()[:760] # After the gt are nan

# Frames 
list_frames = []
for file in sorted(glob.glob('surfer/*png')):
    frame = cv2.imread(file)
#     frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    list_frames.append(frame)
    
create_video(list_frames[:760], "surfer.mp4", fps=24)

### Tracking object

#### Experiment 1

In [4]:
# Initialize kalman filter - top left point
KF_top_left = KalmanFilter(0.1, 1, 1, 1, 0.1,0.1)
# Initialize kalman filter - bottom right point
KF_bottom_right = KalmanFilter(0.1, 1, 1, 1, 0.1,0.1)

In [5]:
# Tracking object
kf_track_frames = []
gt_mask_sequence = []
estimated_mask_sequence = []

total_time = []

for idx, frame in enumerate(list_frames[:760]):
    
    start_time = time.time()
    height, width = frame.shape[:2]
        
    # Object
    top_left = gt[idx][:2]
    bottom_right = gt[idx][2:]

    # Draw a rectangle as the object position
    cv2.rectangle(frame, (int(top_left[0]), int(top_left[1])), (int(bottom_right[0]), int(bottom_right[1])), (0, 255, 0), 2)
    cv2.putText(frame, "Ground truth", (10, 20), 0, 0.5, (0, 255, 0), 1)
    
    # Ground truth mask
    gt_mask = np.zeros((height, width))
    gt_mask[int(gt[idx][1]):int(gt[idx][1]+(gt[idx][3]-gt[idx][1])),int(gt[idx][0]):int(gt[idx][0]+(gt[idx][2]-gt[idx][0]))]=1
    gt_mask_sequence.append(gt_mask)

    # Kalman filter - Predict
    (x_top_left, y_top_left) = KF_top_left.predict()
    (x_bottom_right, y_bottom_right) = KF_bottom_right.predict()
    
    # Kalman filter - Update
    x1_top_left, y1_top_left = KF_top_left.update(top_left).tolist()[0]
    x1_bottom_right, y1_bottom_right = KF_bottom_right.update(bottom_right).tolist()[0]

    # Draw a rectangle as the estimated object position
    cv2.rectangle(frame, (int(x1_top_left), int(y1_top_left)), (int(x1_bottom_right), int(y1_bottom_right)), (0, 0, 255), 2)
    cv2.putText(frame, "Estimated", (10, 40), 0, 0.5, (0, 0, 255), 1)

    # Estimated mask
    estimated_mask = np.zeros((height, width))
    estimated_mask[int(y1_top_left):int(y1_top_left+(y1_bottom_right-y1_top_left)),int(x1_top_left):int(x1_top_left+(x1_bottom_right-x1_top_left))]=1
    estimated_mask_sequence.append(estimated_mask)
    
    kf_track_frames.append(frame)
    total_time.append(round(time.time() - start_time, 3))
        
print(f"Total time: {sum(total_time)}")
print(f"Time per frame: {round(np.mean(total_time),3)}")

Total time: 0.6350000000000005
Time per frame: 0.001


In [6]:
# Saving video
create_video(kf_track_frames, "output_kalman.mp4", fps=24)
# Metrics
extract_metrics(gt_mask_sequence, estimated_mask_sequence)

C:\Users\SOFIA\Anaconda3\lib\site-packages\ipykernel_launcher.py:145: RuntimeWarning: invalid value encountered in double_scalars


Pixel Accuracy: 0.991
Pixel Precision: 0.914
IoU Metric: 0.847


#### Experiment 2

In [7]:
"""
dt: sampling time (time for 1 cycle)
u_x: acceleration in x-direction
u_y: acceleration in y-direction
std_acc: process noise magnitude
x_std_meas: standard deviation of the measurement in x-direction
y_std_meas: standard deviation of the measurement in y-direction
"""

# Initialize kalman filter - top left point
KF_top_left = KalmanFilter(0.5, 1, 1, 1, 0.1,0.1)
# Initialize kalman filter - bottom right point
KF_bottom_right = KalmanFilter(0.5, 1, 1, 1, 0.1,0.1)

In [8]:
# Tracking object
kf_track_frames = []
gt_mask_sequence = []
estimated_mask_sequence = []

total_time = []

for idx, frame in enumerate(list_frames[:760]):
    
    start_time = time.time()
    height, width = frame.shape[:2]
        
    # Object
    top_left = gt[idx][:2]
    bottom_right = gt[idx][2:]

    # Draw a rectangle as the object position
    cv2.rectangle(frame, (int(top_left[0]), int(top_left[1])), (int(bottom_right[0]), int(bottom_right[1])), (0, 255, 0), 2)
    cv2.putText(frame, "Ground truth", (10, 20), 0, 0.5, (0, 255, 0), 1)
    
    # Ground truth mask
    gt_mask = np.zeros((height, width))
    gt_mask[int(gt[idx][1]):int(gt[idx][1]+(gt[idx][3]-gt[idx][1])),int(gt[idx][0]):int(gt[idx][0]+(gt[idx][2]-gt[idx][0]))]=1
    gt_mask_sequence.append(gt_mask)

    # Kalman filter - Predict
    (x_top_left, y_top_left) = KF_top_left.predict()
    (x_bottom_right, y_bottom_right) = KF_bottom_right.predict()
    
    # Kalman filter - Update
    x1_top_left, y1_top_left = KF_top_left.update(top_left).tolist()[0]
    x1_bottom_right, y1_bottom_right = KF_bottom_right.update(bottom_right).tolist()[0]

    # Draw a rectangle as the estimated object position
    cv2.rectangle(frame, (int(x1_top_left), int(y1_top_left)), (int(x1_bottom_right), int(y1_bottom_right)), (0, 0, 255), 2)
    cv2.putText(frame, "Estimated", (10, 40), 0, 0.5, (0, 0, 255), 1)

    # Estimated mask
    estimated_mask = np.zeros((height, width))
    estimated_mask[int(y1_top_left):int(y1_top_left+(y1_bottom_right-y1_top_left)),int(x1_top_left):int(x1_top_left+(x1_bottom_right-x1_top_left))]=1
    estimated_mask_sequence.append(estimated_mask)
    
    kf_track_frames.append(frame)
    total_time.append(round(time.time() - start_time, 3))
        
print(f"Total time: {sum(total_time)}")
print(f"Time per frame: {round(np.mean(total_time),3)}")

# Saving video
create_video(kf_track_frames, "output_kalman_param2.mp4", fps=24)
# Metrics
extract_metrics(gt_mask_sequence, estimated_mask_sequence)

Total time: 0.5530000000000004
Time per frame: 0.001
Pixel Accuracy: 1.0
Pixel Precision: 0.998
IoU Metric: 0.997


#### Experiment 3

In [9]:
"""
dt: sampling time (time for 1 cycle)
u_x: acceleration in x-direction
u_y: acceleration in y-direction
std_acc: process noise magnitude
x_std_meas: standard deviation of the measurement in x-direction
y_std_meas: standard deviation of the measurement in y-direction
"""

# Initialize kalman filter - top left point
KF_top_left = KalmanFilter(0.1, 1, 1, 1, 0.3,0.3)
# Initialize kalman filter - bottom right point
KF_bottom_right = KalmanFilter(0.1, 1, 1, 1, 0.3,0.3)

In [10]:
# Tracking object
kf_track_frames = []
gt_mask_sequence = []
estimated_mask_sequence = []

total_time = []

for idx, frame in enumerate(list_frames[:760]):
    
    start_time = time.time()
    height, width = frame.shape[:2]
        
    # Object
    top_left = gt[idx][:2]
    bottom_right = gt[idx][2:]

    # Draw a rectangle as the object position
    cv2.rectangle(frame, (int(top_left[0]), int(top_left[1])), (int(bottom_right[0]), int(bottom_right[1])), (0, 255, 0), 2)
    cv2.putText(frame, "Ground truth", (10, 20), 0, 0.5, (0, 255, 0), 1)
    
    # Ground truth mask
    gt_mask = np.zeros((height, width))
    gt_mask[int(gt[idx][1]):int(gt[idx][1]+(gt[idx][3]-gt[idx][1])),int(gt[idx][0]):int(gt[idx][0]+(gt[idx][2]-gt[idx][0]))]=1
    gt_mask_sequence.append(gt_mask)

    # Kalman filter - Predict
    (x_top_left, y_top_left) = KF_top_left.predict()
    (x_bottom_right, y_bottom_right) = KF_bottom_right.predict()
    
    # Kalman filter - Update
    x1_top_left, y1_top_left = KF_top_left.update(top_left).tolist()[0]
    x1_bottom_right, y1_bottom_right = KF_bottom_right.update(bottom_right).tolist()[0]

    # Draw a rectangle as the estimated object position
    cv2.rectangle(frame, (int(x1_top_left), int(y1_top_left)), (int(x1_bottom_right), int(y1_bottom_right)), (0, 0, 255), 2)
    cv2.putText(frame, "Estimated", (10, 40), 0, 0.5, (0, 0, 255), 1)

    # Estimated mask
    estimated_mask = np.zeros((height, width))
    estimated_mask[int(y1_top_left):int(y1_top_left+(y1_bottom_right-y1_top_left)),int(x1_top_left):int(x1_top_left+(x1_bottom_right-x1_top_left))]=1
    estimated_mask_sequence.append(estimated_mask)
    
    kf_track_frames.append(frame)
    total_time.append(round(time.time() - start_time, 3))
        
print(f"Total time: {sum(total_time)}")
print(f"Time per frame: {round(np.mean(total_time),3)}")

# Saving video
create_video(kf_track_frames, "output_kalman_param3.mp4", fps=24)
# Metrics
extract_metrics(gt_mask_sequence, estimated_mask_sequence)

Total time: 0.5570000000000004
Time per frame: 0.001
Pixel Accuracy: 0.983
Pixel Precision: 0.835
IoU Metric: 0.738


#### Experiment 4

In [11]:
"""
dt: sampling time (time for 1 cycle)
u_x: acceleration in x-direction
u_y: acceleration in y-direction
std_acc: process noise magnitude
x_std_meas: standard deviation of the measurement in x-direction
y_std_meas: standard deviation of the measurement in y-direction
"""

# Initialize kalman filter - top left point
KF_top_left = KalmanFilter(0.1, 1, 1, 3, 0.1, 0.1)
# Initialize kalman filter - bottom right point
KF_bottom_right = KalmanFilter(0.1, 1, 1, 3, 0.1, 0.1)

In [12]:
# Tracking object
kf_track_frames = []
gt_mask_sequence = []
estimated_mask_sequence = []

total_time = []

for idx, frame in enumerate(list_frames[:760]):
    
    start_time = time.time()
    height, width = frame.shape[:2]
        
    # Object
    top_left = gt[idx][:2]
    bottom_right = gt[idx][2:]

    # Draw a rectangle as the object position
    cv2.rectangle(frame, (int(top_left[0]), int(top_left[1])), (int(bottom_right[0]), int(bottom_right[1])), (0, 255, 0), 2)
    cv2.putText(frame, "Ground truth", (10, 20), 0, 0.5, (0, 255, 0), 1)
    
    # Ground truth mask
    gt_mask = np.zeros((height, width))
    gt_mask[int(gt[idx][1]):int(gt[idx][1]+(gt[idx][3]-gt[idx][1])),int(gt[idx][0]):int(gt[idx][0]+(gt[idx][2]-gt[idx][0]))]=1
    gt_mask_sequence.append(gt_mask)

    # Kalman filter - Predict
    (x_top_left, y_top_left) = KF_top_left.predict()
    (x_bottom_right, y_bottom_right) = KF_bottom_right.predict()
    
    # Kalman filter - Update
    x1_top_left, y1_top_left = KF_top_left.update(top_left).tolist()[0]
    x1_bottom_right, y1_bottom_right = KF_bottom_right.update(bottom_right).tolist()[0]

    # Draw a rectangle as the estimated object position
    cv2.rectangle(frame, (int(x1_top_left), int(y1_top_left)), (int(x1_bottom_right), int(y1_bottom_right)), (0, 0, 255), 2)
    cv2.putText(frame, "Estimated", (10, 40), 0, 0.5, (0, 0, 255), 1)

    # Estimated mask
    estimated_mask = np.zeros((height, width))
    estimated_mask[int(y1_top_left):int(y1_top_left+(y1_bottom_right-y1_top_left)),int(x1_top_left):int(x1_top_left+(x1_bottom_right-x1_top_left))]=1
    estimated_mask_sequence.append(estimated_mask)
    
    kf_track_frames.append(frame)
    total_time.append(round(time.time() - start_time, 3))
        
print(f"Total time: {sum(total_time)}")
print(f"Time per frame: {round(np.mean(total_time),3)}")

# Saving video
create_video(kf_track_frames, "output_kalman_param4.mp4", fps=24)
# Metrics
extract_metrics(gt_mask_sequence, estimated_mask_sequence)

Total time: 0.5490000000000004
Time per frame: 0.001
Pixel Accuracy: 0.997
Pixel Precision: 0.967
IoU Metric: 0.938
